In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

In [3]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()

In [4]:
spilter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=30)
spilted_data = spilter.split_documents(docs)

In [6]:
embedded = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
store_db = Chroma.from_documents(
    documents= spilted_data,
    embedding=embedded
)

In [7]:
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

In [8]:
@tool
def retrival_tool(query:str):
    """
        This tool can help you to retrieve the relevant data of the PDF Documents, and these pdf documents have details about medical reports.
    """
    docs = store_db.similarity_search(
        query=query,
        k=4
    )
    
    context = ""
    for doc in docs:
        context += doc.page_content+"/n/n"
        
    return context

In [9]:
query = "Name of Doc and Pateint"
llm = ChatGroq(
    model="openai/gpt-oss-120b"
)

In [11]:
System_Prompt = """
    You are a helpful assistant that answers questions using retrieved context.
	ALWAYS use the `retriever_tool` tool for questions requiring external knowledge.
"""

In [12]:
agent = create_agent(
    model=llm,
    tools=[retrival_tool],
    system_prompt=System_Prompt
)

In [13]:
query = "What is the name of patient, and what is the name of Doctors"
response = agent.invoke({"messages":[{"role":"user", "content":query}]})

In [16]:
res = response["messages"][-1].content
print(res)

**Patient name:** Ms. Nikita Chudhary  

**Doctors mentioned in the report:**

- **Dr. Nitin Nahar** – the referring physician (shown under “Ref By”).  
- **Dr. Swapnil Gupta, MD, Pathology** – Chief of Laboratory, Dr Lal PathLabs Ltd.  
- **Dr. Kiran Bhargava Pathak, MD, Pathology** – Chief of Laboratory, Dr Lal PathLabs Ltd.  

These are the names that appear in the retrieved medical report.
